#Day3 EASY: Train Your First Models
### SDA AI Bootcamp Supervised Learning for Regression & Classification

**Time:** ~25-30 minutes
**Datasets:** Bike-Share Demand (regression) & Student Pass/Fail (classification) same datasets used in today's slides
**Goal:** get comfortable with the core loop every supervised model follows: **split → train → predict → evaluate**

This notebook uses **one algorithm per task** (Linear Regression, Logistic Regression) so you can focus entirely on the workflow. The Medium and Hard notebooks add more algorithms, cross-validation, and tuning.

Cells marked **`# TODO`** are for you. A collapsed ** Solution** follows each one.


## 1. Setup

In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import mean_squared_error, r2_score, accuracy_score, f1_score

print("Libraries loaded.")


Libraries loaded.


## 2. Regression Task Predict Bike Rental Demand

Load the data. **Note:** `casual` + `registered` add up EXACTLY to `count` (our target) including them as features would be leaking the answer straight into the input. We exclude them below on purpose.

In [2]:
try:
    df = pd.read_csv("https://raw.githubusercontent.com/justmarkham/DAT8/master/data/bikeshare.csv")
    print("Loaded.")
except Exception as e:
    print("Download failed:", e, "\nUpload bikeshare.csv manually:")
    # from google.colab import files
    # uploaded = files.upload()
    # df = pd.read_csv(list(uploaded.keys())[0])

features = ["season", "holiday", "workingday", "weather", "temp", "atemp", "humidity", "windspeed"]
X = df[features]
y = df["count"]
print(X.shape, y.shape)
X.head()


Loaded.
(10886, 8) (10886,)


,season,holiday,workingday,weather,temp,atemp,humidity,windspeed
0,1,0,0,1,9.84,14.395,81,0.0
1,1,0,0,1,9.02,13.635,80,0.0
2,1,0,0,1,9.02,13.635,80,0.0
3,1,0,0,1,9.84,14.395,75,0.0
4,1,0,0,1,9.84,14.395,75,0.0


**`# TODO`** Split into train/test (80/20), then scale the features. Scaling matters less for plain Linear Regression, but it's good habit for every model you'll meet today.

In [3]:
# TODO: split with test_size=0.2, random_state=42
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)
print(X_train_s.shape, X_test_s.shape)


(8708, 8) (2178, 8)


<details><summary> Solution</summary>

```python
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
```
</details>

**`# TODO`** Train a `LinearRegression` model, then generate predictions on the test set.

In [4]:
# TODO: create and fit a LinearRegression model on the SCALED training data
lin_reg = LinearRegression()
lin_reg.fit(X_train_s, y_train)

# TODO: predict on the scaled test data
preds = lin_reg.predict(X_test_s)
preds[:5]


array([315.86982513,  40.92678568, 178.49750562, 280.27174161,
       258.46432297])

<details><summary> Solution</summary>

```python
lin_reg = LinearRegression()
lin_reg.fit(X_train_s, y_train)
preds = lin_reg.predict(X_test_s)
```
</details>

**`# TODO`** Evaluate with RMSE and R² (both introduced in today's lecture).

In [5]:
# TODO: compute RMSE (hint: mean_squared_error(...) ** 0.5) and R2
rmse = mean_squared_error(y_test, preds) ** 0.5
r2 = r2_score(y_test, preds)
print(f"RMSE: {rmse:.2f} bikes/hour")
print(f"R2:   {r2:.3f}")


RMSE: 154.62 bikes/hour
R2:   0.276


<details><summary> Solution</summary>

```python
rmse = mean_squared_error(y_test, preds) ** 0.5
r2 = r2_score(y_test, preds)
```

Compare your RMSE to the lecture's Linear Regression result (154.62) you should get the same number, since it's the same data, split, and model.
</details>

In [6]:
# Self-check
assert 140 < rmse < 170, f"RMSE {rmse:.1f} is outside the expected range -- check your split/scaling."
print("Regression result looks correct.")


Regression result looks correct.


## 3. Classification Task Predict Student Pass/Fail

Same loop, different problem type. We deliberately exclude `G1`/`G2` (earlier-period grades) to make this a genuine **early-warning** prediction, not a copy of mid-year grades.

In [7]:
try:
    sdf = pd.read_csv("https://raw.githubusercontent.com/arunk13/MSDA-Assignments/master/IS607Fall2015/Assignment3/student-mat.csv", sep=";")
    print("Loaded.")
except Exception as e:
    print("Download failed:", e, "\nUpload student-mat.csv manually:")
    # from google.colab import files
    # uploaded = files.upload()
    # sdf = pd.read_csv(list(uploaded.keys())[0], sep=";")

sdf["Pass"] = (sdf["G3"] >= 10).astype(int)
Xs = sdf.drop(columns=["G1", "G2", "G3", "Pass"])
ys = sdf["Pass"]

cat_cols = Xs.select_dtypes(include="object").columns.tolist()
Xs_enc = Xs.copy()
for c in cat_cols:
    Xs_enc[c] = LabelEncoder().fit_transform(Xs_enc[c])

print("Pass rate:", ys.mean().round(3))
Xs_enc.head()


Loaded.
Pass rate: 0.671


,school,sex,age,address,famsize,Pstatus,Medu,Fedu,Mjob,Fjob,...,higher,internet,romantic,famrel,freetime,goout,Dalc,Walc,health,absences
0,0,0,18,1,0,0,4,4,0,4,...,1,0,0,4,3,4,1,1,3,6
1,0,0,17,1,0,1,1,1,0,2,...,1,1,0,5,3,3,1,1,3,4
2,0,0,15,1,1,1,1,1,0,2,...,1,1,0,4,3,2,2,3,3,10
3,0,0,15,1,0,1,4,2,1,3,...,1,1,1,3,2,2,1,1,5,2
4,0,0,16,1,0,1,3,3,2,2,...,1,0,0,4,3,2,1,2,5,4


**`# TODO`** Split (stratified, since ~67/33 isn't perfectly balanced), scale, train a `LogisticRegression`, and predict.

In [8]:
# TODO: stratified split, test_size=0.2, random_state=42
Xs_train, Xs_test, ys_train, ys_test = train_test_split(Xs_enc, ys, test_size=0.2, random_state=42, stratify=ys)

scaler2 = StandardScaler()
Xs_train_s = scaler2.fit_transform(Xs_train)
Xs_test_s = scaler2.transform(Xs_test)

# TODO: train LogisticRegression (max_iter=2000) and predict on the test set
log_reg = LogisticRegression(max_iter=2000)
log_reg.fit(Xs_train_s, ys_train)
clf_preds = log_reg.predict(Xs_test_s)
clf_preds[:10]


array([0, 0, 1, 1, 1, 1, 1, 0, 1, 1])

<details><summary> Solution</summary>

```python
Xs_train, Xs_test, ys_train, ys_test = train_test_split(
    Xs_enc, ys, test_size=0.2, random_state=42, stratify=ys
)
log_reg = LogisticRegression(max_iter=2000)
log_reg.fit(Xs_train_s, ys_train)
clf_preds = log_reg.predict(Xs_test_s)
```
</details>

**`# TODO`** Evaluate with Accuracy and F1-Score.

In [9]:
# TODO: compute accuracy and F1
acc = accuracy_score(ys_test, clf_preds)
f1 = f1_score(ys_test, clf_preds)
print(f"Accuracy: {acc:.3f}")
print(f"F1-Score: {f1:.3f}")


Accuracy: 0.684
F1-Score: 0.783


<details><summary> Solution</summary>

```python
acc = accuracy_score(ys_test, clf_preds)
f1 = f1_score(ys_test, clf_preds)
```

Compare to the lecture's Logistic Regression result (Accuracy=0.684, F1=0.783) same numbers, same reasons.
</details>

In [10]:
# Self-check
assert 0.6 < acc < 0.75, f"Accuracy {acc:.3f} is outside the expected range -- check your split/scaling."
print("Classification result looks correct.")


Classification result looks correct.


---
## Reflection

1. Why did we exclude `casual` and `registered` from the regression features?
2. Why did we use `stratify=ys` for the classification split, but not for the regression split?
3. RMSE and Accuracy are in completely different units why can't you compare a regression score directly to a classification score?

*(1_ Because their sum equals the target count so including them would leak the answer directly into the model inputs

2_ To keep the proportion of classes balanced in classification sets whereas regression deals with continuous numbers where stratification isn't needed

3_ Because RMSE measures numerical error magnitude while Accuracy measures a percentage of correct categorical predictions making their units and goals entirely different
)*

---
### Next up
**Medium** adds 3 more algorithms per task, a real comparison table, and cross-validation.

### Dataset credit
Bike-share dataset via `justmarkham/DAT8`. Student Performance dataset (Cortez & Silva, 2008) via UCI Machine Learning Repository.
